# Evaluating Langfuse Traces with fasteval — Overview

This notebook walks you through setting up a complete quality evaluation pipeline using **fasteval** and **Langfuse**.

**What you'll learn:**

1. **The Evaluation Loop** — How online and offline evaluation work together
2. **Installation** — Install `fasteval-core` and `fasteval-langfuse`
3. **API Key Setup** — Configure Langfuse and OpenAI credentials (Colab-friendly)
4. **Code Configuration** — Use `LangfuseConfig` for full control over behavior
5. **Verify Connection** — Confirm your Langfuse project is reachable
6. **Quick Taste** — Run a minimal trace evaluation to see it all work

Every cell is **runnable**. Cells that call LLM-as-judge metrics require an OpenAI API key.

---

## The Evaluation Loop

Effective LLM engineering requires a continuous feedback loop between production and development:

```
┌──────────────────────────────────────────────────────────────┐
│  1. ONLINE EVALUATION        2. IDENTIFY ISSUES             │
│  Sample live traces     →    Filter by low scores           │
│  (@langfuse_traces)          (fasteval_correctness < 0.7)   │
│       │                             │                       │
│       │                             ▼                       │
│  5. DEPLOY                   3. BUILD DATASETS              │
│  CI/CD gates pass       ←    Promote traces to golden sets  │
│       ▲                             │                       │
│       │                             ▼                       │
│  4. OFFLINE EVALUATION                                      │
│  Test against golden sets                                   │
│  (@langfuse_dataset)                                        │
└──────────────────────────────────────────────────────────────┘
```

This cookbook covers the full cycle across 4 notebooks:

| Notebook | Step | What It Covers |
|----------|------|----------------|
| **This one** | Setup | Installation, configuration, connection verification |
| [Basic Trace Evaluation](https://colab.research.google.com/github/intuit/fasteval/blob/main/docs/notebooks/langfuse-trace-evaluation/basic-trace-evaluation.ipynb) | Steps 1-2 | Evaluate production traces, view scores in Langfuse |
| [Sampling Strategies](https://colab.research.google.com/github/intuit/fasteval/blob/main/docs/notebooks/langfuse-trace-evaluation/sampling-strategies.ipynb) | Step 1 | Optimize costs with smart sampling, scheduling, automation |
| [Dataset Regression Testing](https://colab.research.google.com/github/intuit/fasteval/blob/main/docs/notebooks/langfuse-trace-evaluation/dataset-regression-testing.ipynb) | Steps 3-5 | Golden datasets, A/B testing, Langfuse Experiments, CI/CD |

---

## 1. Installation

In [ ]:
!pip install -q fasteval-core fasteval-langfuse

In [ ]:
import fasteval as fe
import fasteval_langfuse

print(f"fasteval version:          {fe.__version__}")
print(f"fasteval-langfuse version: {fasteval_langfuse.__version__}")

---

## 2. API Key Setup

You need three keys:

| Key | Where to get it | Purpose |
|-----|----------------|----------|
| `LANGFUSE_PUBLIC_KEY` | Langfuse > Settings > API Keys | Authenticate reads (traces, datasets) |
| `LANGFUSE_SECRET_KEY` | Langfuse > Settings > API Keys | Authenticate writes (scores) |
| `OPENAI_API_KEY` | platform.openai.com | Power the LLM judge for evaluation metrics |

> **Colab Note:** The cell below tries to load keys from Colab Secrets first (recommended). Click the **key icon** in the left sidebar to add secrets without exposing them in code.

In [ ]:
import os

# --- Langfuse Credentials ---
#
# Option 1: Colab Secrets (recommended — click the key icon in sidebar)
# Option 2: Set directly (uncomment and paste your keys)
#
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://cloud.langfuse.com"
# os.environ["OPENAI_API_KEY"] = "sk-..."

try:
    from google.colab import userdata
    os.environ["LANGFUSE_PUBLIC_KEY"] = userdata.get("LANGFUSE_PUBLIC_KEY")
    os.environ["LANGFUSE_SECRET_KEY"] = userdata.get("LANGFUSE_SECRET_KEY")
    os.environ["LANGFUSE_HOST"] = userdata.get("LANGFUSE_HOST")
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("All keys loaded from Colab Secrets.")
except (ImportError, Exception):
    pass

# Verify keys are present
required = ["LANGFUSE_PUBLIC_KEY", "LANGFUSE_SECRET_KEY", "OPENAI_API_KEY"]
for key in required:
    status = "SET" if os.environ.get(key) else "MISSING"
    print(f"  {key}: {status}")

host = os.environ.get("LANGFUSE_HOST", "https://cloud.langfuse.com")
print(f"  LANGFUSE_HOST: {host}")

---

## 3. Configuration

Use `configure_langfuse()` to control how the integration behaves. This is typically done once, at the start of your test session (or in `conftest.py` for pytest).

**Key options:**

| Parameter | Default | Description |
|-----------|---------|-------------|
| `auto_push_scores` | `True` | Push evaluation results back to Langfuse as scores |
| `score_name_prefix` | `"fasteval_"` | Prefix for score names (e.g., `fasteval_correctness`) |
| `batch_size` | `50` | Fetch traces in batches of this size |
| `max_parallel_evals` | `5` | Concurrent evaluations |
| `retry_on_failure` | `True` | Retry score pushes on network errors |

> **Colab Note:** If you set environment variables above, you can skip `public_key`, `secret_key`, and `host` — they will be read from the environment automatically.

In [ ]:
from fasteval_langfuse import configure_langfuse, LangfuseConfig

config = LangfuseConfig(
    # Credentials — omit if env vars are set
    # public_key="pk-lf-...",
    # secret_key="sk-lf-...",
    # host="https://cloud.langfuse.com",

    # Score Reporting
    auto_push_scores=True,           # Push results back to Langfuse
    score_name_prefix="fasteval_",   # e.g., "fasteval_correctness"
    retry_on_failure=True,           # Retry on network errors

    # Performance
    batch_size=50,                   # Traces per batch
    max_parallel_evals=5,            # Concurrent evaluations
)

configure_langfuse(config)
print("Configuration applied.")
print(f"  auto_push_scores:  {config.auto_push_scores}")
print(f"  score_name_prefix: {config.score_name_prefix}")
print(f"  batch_size:        {config.batch_size}")

---

## 4. Verify Connection

Before running evaluations, let's confirm we can reach the Langfuse API.

> **Colab Note:** If this cell fails, double-check that your `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY`, and `LANGFUSE_HOST` are correct. For self-hosted instances, ensure the host URL is accessible from Colab.

In [ ]:
from langfuse import Langfuse

langfuse = Langfuse()

# Quick health check — fetch project info
try:
    projects = langfuse.client.projects.get()
    print("Connection successful!")
    print(f"  Projects available: {[p.name for p in projects.data]}")
except Exception as e:
    print(f"Connection failed: {e}")
    print("Check your API keys and host URL.")

---

## 5. Quick Taste — Your First Trace Evaluation

Let's run a minimal evaluation to see the full flow: fetch traces, evaluate with an LLM judge, and push scores back to Langfuse.

This example:
- Fetches up to **3 traces** from the last 24 hours
- Evaluates each for **correctness** (is the response appropriate for the input?)
- Pushes the score back to Langfuse with the `fasteval_` prefix

> **Colab Note:** If your project has no traces yet, this will fetch 0 items and pass. You can [create sample traces](https://langfuse.com/docs/get-started) using the Langfuse SDK first.

In [ ]:
from fasteval_langfuse import langfuse_traces


@fe.correctness(threshold=0.7)
@langfuse_traces(
    time_range="last_24h",
    limit=3,
)
def test_quick_taste(trace_id, input, output, context, metadata):
    """Evaluate a small sample of recent traces."""
    print(f"  Trace {trace_id[:12]}...")
    print(f"    Input:  {str(input)[:80]}")
    print(f"    Output: {str(output)[:80]}")
    fe.score(output, input=input)


# Run the evaluation
print("Fetching traces and evaluating...\n")
test_quick_taste()

If scores were pushed, you can now go to **Langfuse > Traces** and see new `fasteval_correctness` scores on the evaluated traces.

---

## 6. fasteval vs. Langfuse Native Evaluators

Langfuse offers built-in LLM-as-a-Judge evaluators that run in their cloud. When should you use fasteval instead?

| Feature | fasteval | Langfuse Native |
|---------|----------|-----------------|
| **Runs in** | Your environment (local / CI) | Langfuse cloud |
| **CI/CD** | Tests fail the build directly | Webhooks/polling needed |
| **Metrics** | 30+ (RAG, conversation, tools, vision) | Core set |
| **Metric stacking** | Combine deterministic + LLM metrics | Mostly LLM-based |
| **Judge model** | You choose (GPT-4o, Claude, etc.) | Managed by Langfuse |
| **Cost** | Pay your LLM provider directly | Langfuse usage-based |

**Best practice:** Use both! Langfuse native evaluators for lightweight real-time monitoring, fasteval for deep-dive analysis and CI/CD gates.

---

## Next Steps

Now that you're set up, dive into the evaluation loop:

| Notebook | What you'll learn |
|----------|-------------------|
| [Basic Trace Evaluation](https://colab.research.google.com/github/intuit/fasteval/blob/main/docs/notebooks/langfuse-trace-evaluation/basic-trace-evaluation.ipynb) | Fetch and evaluate production traces, view scores in Langfuse, stack metrics |
| [Sampling Strategies](https://colab.research.google.com/github/intuit/fasteval/blob/main/docs/notebooks/langfuse-trace-evaluation/sampling-strategies.ipynb) | All 5 sampling strategies, scheduling patterns, GitHub Actions automation |
| [Dataset Regression Testing](https://colab.research.google.com/github/intuit/fasteval/blob/main/docs/notebooks/langfuse-trace-evaluation/dataset-regression-testing.ipynb) | Golden datasets, A/B testing, Langfuse Experiments, CI/CD gates |

For the full API reference, see the [fasteval-langfuse plugin docs](https://fasteval.dev/docs/plugins/langfuse/overview).